# Particle-level Selection

In [1]:
# Particle-level summary tables for requested MG5 and Sherpa samples
import os
import re
import glob
import ROOT
import numpy as np
import concurrent.futures

from tqdm import tqdm
from pathlib import Path

from particle_level_selection import (
    count_stage_range,
    sherpa_weighted_stage_sums_for_file,
 )

ROOT.gROOT.ProcessLine('.include /usr/local/Delphes-3.4.2/')
ROOT.gROOT.ProcessLine('.include /usr/local/Delphes-3.4.2/external/')
ROOT.gInterpreter.Declare('#include "/usr/local/Delphes-3.4.2/classes/DelphesClasses.h"')
ROOT.gSystem.Load('/usr/local/Delphes-3.4.2/install/lib/libDelphes')

STAGES = ["Total", "Lepton cut", "MET cut", "Jet cut"]
STAGE_MAP = {"Total": 0, "Lepton cut": 1, "MET cut": 2, "Jet cut": 3}

# Parallel settings (tune by CPU and I/O bandwidth)
N_WORKERS = max(1, os.cpu_count() // 2)
MG_CHUNK_SIZE = 1000


def get_cross_section_from_banner_fb(banner_path):
    pattern = re.compile(r"#\s+Integrated weight \(pb\)\s+:\s+([0-9]+\.[0-9eE+-]+)")
    with open(banner_path, "r") as f:
        for line in f:
            m = pattern.match(line)
            if m:
                return float(m.group(1)) * 1000.0
    raise RuntimeError(f"Cannot parse integrated weight from {banner_path}")


def open_delphes_tree(root_path):
    f = ROOT.TFile.Open(root_path)
    if not f or f.IsZombie():
        return None, None
    tree = f.Get("Delphes")
    if tree is None or not hasattr(tree, "GetEntries"):
        f.Close()
        return None, None
    return f, tree


def _chunk_ranges(n_entries, chunk_size):
    ranges = []
    for start in range(0, n_entries, chunk_size):
        end = min(start + chunk_size, n_entries)
        ranges.append((start, end))
    return ranges


def root_cutflow_counts(root_path, workers=N_WORKERS, chunk_size=MG_CHUNK_SIZE):
    f, tree = open_delphes_tree(root_path)
    if tree is None:
        raise RuntimeError(f"Cannot read Delphes tree from {root_path}")

    n_entries = int(tree.GetEntries())
    f.Close()

    chunks = _chunk_ranges(n_entries, chunk_size)
    if len(chunks) == 0:
        return {s: 0 for s in STAGES}

    counts = {s: 0 for s in STAGES}
    max_workers = max(1, min(int(workers), len(chunks)))

    with concurrent.futures.ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(count_stage_range, root_path, start, end)
            for start, end in chunks
        ]
        for fut in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc=f"MG chunks: {Path(root_path).name}"):
            part = fut.result()
            for s in STAGES:
                counts[s] += int(part[s])

    return counts


def extract_pol_weights(event):
    w_dict = dict(zip(event.weight_names, event.weights))
    w_nom = float(w_dict.get("Weight", 1.0))
    is_w_plus = any("W+" in n for n in event.weight_names)
    if is_w_plus:
        w_ll = float(w_dict.get("PolWeight_COM.W+.0_W+.0", 0.0))
        w_lt = float(w_dict.get("PolWeight_COM.W+.0_W+.T", 0.0)) + float(w_dict.get("PolWeight_COM.W+.T_W+.0", 0.0))
        w_tt = float(w_dict.get("PolWeight_COM.W+.T_W+.T", 0.0))
    else:
        w_ll = float(w_dict.get("PolWeight_COM.W-.0_W-.0", 0.0))
        w_lt = float(w_dict.get("PolWeight_COM.W-.0_W-.T", 0.0)) + float(w_dict.get("PolWeight_COM.W-.T_W-.0", 0.0))
        w_tt = float(w_dict.get("PolWeight_COM.W-.T_W-.T", 0.0))
    return w_nom, w_ll, w_lt, w_tt


def sherpa_stage_table(hepmc_dir, root_dir, workers=N_WORKERS):
    sums = {s: {"nominal": 0.0, "LL": 0.0, "LT": 0.0, "TT": 0.0} for s in STAGES}
    xsec_values = []

    hepmc_files = sorted(glob.glob(os.path.join(hepmc_dir, "*.hepmc")))
    pairs = []
    for hepmc_path in hepmc_files:
        base = Path(hepmc_path).stem
        root_path = os.path.join(root_dir, f"{base}.root")
        if os.path.exists(root_path):
            pairs.append((hepmc_path, root_path))

    if not pairs:
        raise RuntimeError(f"No matched HepMC/ROOT pairs found in {hepmc_dir}")

    max_workers = max(1, min(int(workers), len(pairs)))
    skipped_files = 0
    with concurrent.futures.ProcessPoolExecutor(max_workers=max_workers) as executor:
        future_map = {
            executor.submit(sherpa_weighted_stage_sums_for_file, hepmc_path, root_path): (hepmc_path, root_path)
            for hepmc_path, root_path in pairs
        }
        for fut in tqdm(concurrent.futures.as_completed(future_map), total=len(future_map), desc=f"Sherpa pairs: {Path(hepmc_dir).name}"):
            hepmc_path, root_path = future_map[fut]
            try:
                res = fut.result()
            except Exception as exc:
                skipped_files += 1
                print(f"[skip] worker failed for {root_path}: {exc}")
                continue

            if res.get("skipped", False):
                skipped_files += 1
                reason = res.get("reason", "unknown")
                print(f"[skip] {root_path}: {reason}")
                continue

            if res["total_xsec_pb"] > 0:
                xsec_values.append(float(res["total_xsec_pb"]))
            part_sums = res["sums"]
            for s in STAGES:
                sums[s]["nominal"] += float(part_sums[s]["nominal"])
                sums[s]["LL"] += float(part_sums[s]["LL"])
                sums[s]["LT"] += float(part_sums[s]["LT"])
                sums[s]["TT"] += float(part_sums[s]["TT"])

    if skipped_files > 0:
        print(f"[info] skipped {skipped_files} corrupted/unreadable Sherpa ROOT files in {hepmc_dir}")

    total_nominal = sums["Total"]["nominal"]
    if total_nominal <= 0.0:
        raise RuntimeError(f"Total nominal weight is zero for {hepmc_dir}")

    total_xsec_pb = float(np.mean(xsec_values)) if xsec_values else 0.0
    scale_fb = total_xsec_pb * 1000.0 / total_nominal if total_xsec_pb > 0 else 0.0

    table = {}
    for s in STAGES:
        ll = scale_fb * sums[s]["LL"]
        lt = scale_fb * sums[s]["LT"]
        tt = scale_fb * sums[s]["TT"]
        table[s] = {"LL": ll, "LT": lt, "TT": tt, "Total": ll + lt + tt}
    return table


def combine_tables(a, b):
    out = {}
    for s in STAGES:
        ll = a[s]["LL"] + b[s]["LL"]
        lt = a[s]["LT"] + b[s]["LT"]
        tt = a[s]["TT"] + b[s]["TT"]
        out[s] = {"LL": ll, "LT": lt, "TT": tt, "Total": ll + lt + tt}
    return out


def latex_table(table, thesis=(0.285, 0.909, 1.778)):
    tll, tlt, ttt = thesis
    ttot = tll + tlt + ttt
    jet = table["Jet cut"]
    fll = 100.0 * jet["LL"] / jet["Total"] if jet["Total"] > 0 else 0.0
    flt = 100.0 * jet["LT"] / jet["Total"] if jet["Total"] > 0 else 0.0
    ftt = 100.0 * jet["TT"] / jet["Total"] if jet["Total"] > 0 else 0.0
    tfll = 100.0 * tll / ttot
    tflt = 100.0 * tlt / ttot
    tftt = 100.0 * ttt / ttot

    print("\\begin{tabular}{l|rrrr}")
    print("Cut & LL [fb] & LT [fb] & TT [fb] & Total [fb] \\\\ \\hline")
    for s in STAGES:
        r = table[s]
        print(f"{s:<18} & {r['LL']:.3f} & {r['LT']:.3f} & {r['TT']:.3f} & {r['Total']:.3f} \\\\")
    print("\\hline")
    print(f"Jet cut (Thesis)   & {tll:.3f} & {tlt:.3f} & {ttt:.3f} & {ttot:.3f} \\\\")
    print("\\hline")
    print(f"Fraction           & {fll:.1f}\\% & {flt:.1f}\\% & {ftt:.1f}\\% & 100\\% \\\\")
    print(f"Fraction (Thesis)  & {tfll:.1f}\\% & {tflt:.1f}\\% & {tftt:.1f}\\% & 100\\%")
    print("\\end{tabular}")

Welcome to JupyROOT 6.20/08


In [2]:
# MadGraph samples
mg5_base = Path("./MG5")
mg_dirs = {
    "LL": mg5_base / "EW_WWjj_LL-WW_cmf-NNPDF30_nlo_as_0118",
    "LT": mg5_base / "EW_WWjj_LT-WW_cmf-NNPDF30_nlo_as_0118",
    "TT": mg5_base / "EW_WWjj_TT-WW_cmf-NNPDF30_nlo_as_0118",
}

mg_stage_pol = {}
for pol, d in mg_dirs.items():
    root_path = d / "Events" / "run_02" / "tag_1_delphes_events.root"
    banner_path = d / "Events" / "run_02" / "run_02_tag_1_banner.txt"
    counts = root_cutflow_counts(str(root_path), workers=N_WORKERS, chunk_size=MG_CHUNK_SIZE)
    sigma = get_cross_section_from_banner_fb(str(banner_path))
    n_total = counts["Total"]
    mg_stage_pol[pol] = {s: sigma * counts[s] / n_total for s in STAGES}

mg_table = {}
for s in STAGES:
    ll = mg_stage_pol["LL"][s]
    lt = mg_stage_pol["LT"][s]
    tt = mg_stage_pol["TT"][s]
    mg_table[s] = {"LL": ll, "LT": lt, "TT": tt, "Total": ll + lt + tt}

MG chunks: tag_1_delphes_events.root: 100%|██████████| 100/100 [00:17<00:00,  5.81it/s]
Warning in <TStreamerInfo::BuildCheck>: 
   The StreamerInfo of class Track read from file MG5/EW_WWjj_LL-WW_cmf-NNPDF30_nlo_as_0118/Events/run_02/tag_1_delphes_events.root
   has the same version (=3) as the active class but a different checksum.
   You should update the version to ClassDef(Track,4).
   Do not try to write objects with the current class definition,
   the files will not be readable.

Warning in <TStreamerInfo::CompareContent>: The following data member of
the in-memory layout version 3 of class 'Track' is missing from 
the on-file layout version 3:
   float C; //
Warning in <TStreamerInfo::CompareContent>: The following data member of
the in-memory layout version 3 of class 'Track' is missing from 
the on-file layout version 3:
   float Mass; //
Warning in <TStreamerInfo::CompareContent>: The following data member of
the in-memory layout version 3 of class 'Track' is missing from 


In [3]:
print("MadGraph particle-level table:")
latex_table(mg_table, thesis=(0.277, 0.874, 1.744))

MadGraph particle-level table:
\begin{tabular}{l|rrrr}
Cut & LL [fb] & LT [fb] & TT [fb] & Total [fb] \\ \hline
Total              & 0.383 & 1.188 & 2.185 & 3.755 \\
Lepton cut         & 0.267 & 0.850 & 1.665 & 2.782 \\
MET cut            & 0.260 & 0.834 & 1.636 & 2.731 \\
Jet cut            & 0.205 & 0.623 & 1.129 & 1.957 \\
\hline
Jet cut (Thesis)   & 0.277 & 0.874 & 1.744 & 2.895 \\
\hline
Fraction           & 10.5\% & 31.9\% & 57.7\% & 100\% \\
Fraction (Thesis)  & 9.6\% & 30.2\% & 60.2\% & 100\%
\end{tabular}


In [4]:
# Sherpa samples requested by user
wp_table = sherpa_stage_table("./Sherpa/EW_WpWpjj/hepmc_data", "./Sherpa/EW_WpWpjj/delphes_root", workers=N_WORKERS)
wm_table = sherpa_stage_table("./Sherpa/EW_WmWmjj/hepmc_data", "./Sherpa/EW_WmWmjj/delphes_root", workers=N_WORKERS)
sherpa_table = combine_tables(wp_table, wm_table)

Sherpa pairs: hepmc_data: 100%|██████████| 100/100 [00:47<00:00,  2.09it/s]


In [5]:
print("\nSherpa particle-level table (WmWm + WpWp):")
latex_table(sherpa_table, thesis=(0.285, 0.909, 1.778))


Sherpa particle-level table (WmWm + WpWp):
\begin{tabular}{l|rrrr}
Cut & LL [fb] & LT [fb] & TT [fb] & Total [fb] \\ \hline
Total              & 0.729 & 2.252 & 3.989 & 6.970 \\
Lepton cut         & 0.291 & 0.906 & 1.746 & 2.943 \\
MET cut            & 0.240 & 0.786 & 1.522 & 2.548 \\
Jet cut            & 0.188 & 0.569 & 1.021 & 1.777 \\
\hline
Jet cut (Thesis)   & 0.285 & 0.909 & 1.778 & 2.972 \\
\hline
Fraction           & 10.6\% & 32.0\% & 57.5\% & 100\% \\
Fraction (Thesis)  & 9.6\% & 30.6\% & 59.8\% & 100\%
\end{tabular}


In [6]:
# Single combined LaTeX table: left = MadGraph, right = Sherpa
if "mg_table" not in globals() or "sherpa_table" not in globals():
    raise RuntimeError("Please run the cells that build mg_table and sherpa_table first.")

nl = "\\\\"


def _fmt_row(cut_name):
    mg = mg_table[cut_name]
    sh = sherpa_table[cut_name]
    return (
        f"{cut_name:<18} "
        f"& {mg['LL']:.3f} & {mg['LT']:.3f} & {mg['TT']:.3f} & {mg['Total']:.3f} "
        f"& {sh['LL']:.3f} & {sh['LT']:.3f} & {sh['TT']:.3f} & {sh['Total']:.3f} {nl}"
    )


# Thesis reference rows
mg_tll, mg_tlt, mg_ttt = (0.277, 0.874, 1.744)
sh_tll, sh_tlt, sh_ttt = (0.285, 0.909, 1.778)
mg_ttot = mg_tll + mg_tlt + mg_ttt
sh_ttot = sh_tll + sh_tlt + sh_ttt

mg_jet = mg_table["Jet cut"]
sh_jet = sherpa_table["Jet cut"]

mg_fll = 100.0 * mg_jet["LL"] / mg_jet["Total"] if mg_jet["Total"] > 0 else 0.0
mg_flt = 100.0 * mg_jet["LT"] / mg_jet["Total"] if mg_jet["Total"] > 0 else 0.0
mg_ftt = 100.0 * mg_jet["TT"] / mg_jet["Total"] if mg_jet["Total"] > 0 else 0.0

sh_fll = 100.0 * sh_jet["LL"] / sh_jet["Total"] if sh_jet["Total"] > 0 else 0.0
sh_flt = 100.0 * sh_jet["LT"] / sh_jet["Total"] if sh_jet["Total"] > 0 else 0.0
sh_ftt = 100.0 * sh_jet["TT"] / sh_jet["Total"] if sh_jet["Total"] > 0 else 0.0

mg_tfll = 100.0 * mg_tll / mg_ttot if mg_ttot > 0 else 0.0
mg_tflt = 100.0 * mg_tlt / mg_ttot if mg_ttot > 0 else 0.0
mg_tftt = 100.0 * mg_ttt / mg_ttot if mg_ttot > 0 else 0.0

sh_tfll = 100.0 * sh_tll / sh_ttot if sh_ttot > 0 else 0.0
sh_tflt = 100.0 * sh_tlt / sh_ttot if sh_ttot > 0 else 0.0
sh_tftt = 100.0 * sh_ttt / sh_ttot if sh_ttot > 0 else 0.0

print("\\begin{tabular}{l|rrrr|rrrr}")
print("\\hline")
print(f"Cut & \\multicolumn{{4}}{{c|}}{{MadGraph [fb]}} & \\multicolumn{{4}}{{c}}{{Sherpa [fb]}} {nl}")
print(f" & LL & LT & TT & Total & LL & LT & TT & Total {nl} \\hline")

for s in STAGES:
    print(_fmt_row(s))

print("\\hline")
print(
    f"Jet cut (Thesis) "
    f"& {mg_tll:.3f} & {mg_tlt:.3f} & {mg_ttt:.3f} & {mg_ttot:.3f} "
    f"& {sh_tll:.3f} & {sh_tlt:.3f} & {sh_ttt:.3f} & {sh_ttot:.3f} {nl}"
)
print("\\hline")
print(
    f"Fraction "
    f"& {mg_fll:.1f}\\% & {mg_flt:.1f}\\% & {mg_ftt:.1f}\\% & 100\\% "
    f"& {sh_fll:.1f}\\% & {sh_flt:.1f}\\% & {sh_ftt:.1f}\\% & 100\\% {nl}"
)
print(
    f"Fraction (Thesis) "
    f"& {mg_tfll:.1f}\\% & {mg_tflt:.1f}\\% & {mg_tftt:.1f}\\% & 100\\% "
    f"& {sh_tfll:.1f}\\% & {sh_tflt:.1f}\\% & {sh_tftt:.1f}\\% & 100\\% {nl}"
)
print("\\hline")
print("\\end{tabular}")

\begin{tabular}{l|rrrr|rrrr}
\hline
Cut & \multicolumn{4}{c|}{MadGraph [fb]} & \multicolumn{4}{c}{Sherpa [fb]} \\
 & LL & LT & TT & Total & LL & LT & TT & Total \\ \hline
Total              & 0.383 & 1.188 & 2.185 & 3.755 & 0.729 & 2.252 & 3.989 & 6.970 \\
Lepton cut         & 0.267 & 0.850 & 1.665 & 2.782 & 0.291 & 0.906 & 1.746 & 2.943 \\
MET cut            & 0.260 & 0.834 & 1.636 & 2.731 & 0.240 & 0.786 & 1.522 & 2.548 \\
Jet cut            & 0.205 & 0.623 & 1.129 & 1.957 & 0.188 & 0.569 & 1.021 & 1.777 \\
\hline
Jet cut (Thesis) & 0.277 & 0.874 & 1.744 & 2.895 & 0.285 & 0.909 & 1.778 & 2.972 \\
\hline
Fraction & 10.5\% & 31.9\% & 57.7\% & 100\% & 10.6\% & 32.0\% & 57.5\% & 100\% \\
Fraction (Thesis) & 9.6\% & 30.2\% & 60.2\% & 100\% & 9.6\% & 30.6\% & 59.8\% & 100\% \\
\hline
\end{tabular}
